# annstat — GPU experiment suite

Runs E1, E1b, E2, E3, E5, E6, E6b, E7 and saves every result to Google Drive (`MyDrive/annstat_results`).

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Run the cells in order. Start with `PRESET = "quick"` (minutes) to check everything works, then switch to `"full"` (~1–1.5 h on a T4).
3. If the session disconnects, re-run: finished experiments are skipped (set `FORCE = "1"` to recompute).

In [ ]:
# 1) Mount Drive, fetch the package from GitHub
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
REPO_URL = "https://github.com/yamenetoo/Artificial-Neural-Network-Analysis-Using-Statistical-Approaches.git"
BRANCH   = "annstat-package"          # change to "main" after merging
REPO     = "/content/annstat_repo"
if os.path.exists(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull", "-q"], capture_output=True, text=True).stdout or "repo updated")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
!pip -q install tabulate scikit-learn scipy pandas matplotlib
!nvidia-smi -L

In [ ]:
# 2) Configuration
import os
os.environ["ANNSTAT_PRESET"] = "quick"      # "quick" (smoke test) or "full" (paper scale)
os.environ["ANNSTAT_FORCE"]  = "0"          # "1" recomputes finished experiments
os.environ["ANNSTAT_RUN"]    = "E1,E1b,E3,E5,E6,E6b,E2,E7"
os.environ["ANNSTAT_OUT"]    = "/content/drive/MyDrive/annstat_results"

In [ ]:
# 3) Run everything (results are written to Drive after each experiment)
%run /content/annstat_repo/experiments/gpu_suite.py

In [ ]:
# 4) Quick look at the main results
import pandas as pd, os
OUT = os.environ["ANNSTAT_OUT"]
for f in ["e7_ranks", "e7_pairs_holm", "e1b_clt_law", "e5_rank_vs_width", "e8_bound_gpu"]:
    p = f"{OUT}/{f}.csv"
    if os.path.exists(p):
        print("\n==", f); display(pd.read_csv(p).head(30))

In [ ]:
# 5) (optional) download everything as one zip
import shutil
shutil.make_archive("/content/annstat_results", "zip", OUT)
from google.colab import files
files.download("/content/annstat_results.zip")